Retail Sales ETL:-
----------------------

You receive a CSV file containing approximately 50 MB of retail transaction data.

The source contains:

order_id
order_date
customer_id
customer_name
product_id
product_name
category
store_id
store_name
city
state
quantity
unit_price
discount_pct

Requirements:-
------------------
1. Read the source CSV.

Display:

First 10 records
Schema
Number of records

2. Create gross_amount.

Formula:

quantity × unit_price

3. Create discount_amount.

Formula:

gross_amount × discount_pct / 100

4. Create final_amount.

Formula:

gross_amount - discount_amount

5. Rename columns.

customer_id     → customer_key
product_id      → product_key
store_id        → store_key
unit_price      → selling_price
discount_pct    → discount_percentage

6. Create store-level sales summary.

For every store calculate:

total_orders
total_quantity
total_gross_sales
total_discount
total_sales
average_sales
minimum_sales
maximum_sales

7. Create category-level summary.

For every category calculate:

total_quantity
total_sales
average_sales

8. Create product-level summary.

For every product calculate:

total_quantity
total_sales

Sort products by highest sales.

9. Write the processed transaction data.

Output:

processed_sales/

Format:

CSV

10. Write the store summary.

Output:

store_sales_summary/


In [6]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Retal_data_processing").getOrCreate()
from pyspark.sql.types import StructField, StructType, StringType, IntegerType, DoubleType
from pyspark.sql.functions import*

In [9]:
# 1. Read the source CSV.
# Display:
# First 10 records
# Schema
# Number of records

sales_schema = StructType([ StructField('order_id', StringType()),
                            StructField('order_date', DoubleType()),
                            StructField('customer_id', StringType()),
                            StructField('customer_name', StringType()),
                            StructField('product_id', StringType()),
                            StructField('product_name', StringType()),
                            StructField('category', StringType()),
                            StructField('store_id', StringType()),
                            StructField('store_name', StringType()),
                            StructField('city', StringType()),
                            StructField('state', StringType()),
                            StructField('quantity', IntegerType()),
                            StructField('unit_price', DoubleType()),
                            StructField('discount_pct', IntegerType())])

input_df=spark.read.csv('retail_sales_dataset.csv',header = True, schema = sales_schema)

# input_df=spark.read.csv('retail_sales_dataset.csv',header = True, inferSchema = True)
# ---------->
# inferSchema = True - checks every column and row and the datatype of elements & gives us the suitable data type for each column field
# In realtime we should not use inferSchema, it's not ideal because, Spark visits every row and checks their datatype and
# if some few records contains different datatype then the whole field of element could be considered as the datatype of those few records which is wrong
# i.e in a particular single column, most of the values could be of int data type, while in that same column, a few values could be of double data type
# inferSchema = Data Type Validation

input_df.show(10, False)
input_df.printSchema()

+---------+----------+-----------+---------------+----------+------------------+-----------------+--------+-------------------------+-----------+-----+--------+----------+------------+
|order_id |order_date|customer_id|customer_name  |product_id|product_name      |category         |store_id|store_name               |city       |state|quantity|unit_price|discount_pct|
+---------+----------+-----------+---------------+----------+------------------+-----------------+--------+-------------------------+-----------+-----+--------+----------+------------+
|ORD002629|NULL      |CUST00711  |Timothy Scott  |PROD00049 |Puzzle 1000pc     |Toys & Games     |STORE023|Las Vegas Mall Store     |Las Vegas  |NV   |9       |101.58    |0           |
|ORD004246|NULL      |CUST00138  |William Ramirez|PROD00105 |Board Game        |Toys & Games     |STORE013|Fort Worth Mall Store    |Fort Worth |TX   |5       |249.78    |10          |
|ORD001583|NULL      |CUST00049  |Joseph Williams|PROD00047 |Bicycle Helmet

In [10]:
# 2. Create a new column as 'gross_amount'.
# Formula: quantity × unit_price

gross_df = input_df.withColumn('gross_amount', input_df.quantity * input_df.unit_price)
gross_df.show(10, False)
gross_df.printSchema()


+---------+----------+-----------+---------------+----------+------------------+-----------------+--------+-------------------------+-----------+-----+--------+----------+------------+------------+
|order_id |order_date|customer_id|customer_name  |product_id|product_name      |category         |store_id|store_name               |city       |state|quantity|unit_price|discount_pct|gross_amount|
+---------+----------+-----------+---------------+----------+------------------+-----------------+--------+-------------------------+-----------+-----+--------+----------+------------+------------+
|ORD002629|NULL      |CUST00711  |Timothy Scott  |PROD00049 |Puzzle 1000pc     |Toys & Games     |STORE023|Las Vegas Mall Store     |Las Vegas  |NV   |9       |101.58    |0           |914.22      |
|ORD004246|NULL      |CUST00138  |William Ramirez|PROD00105 |Board Game        |Toys & Games     |STORE013|Fort Worth Mall Store    |Fort Worth |TX   |5       |249.78    |10          |1248.9      |
|ORD001583

In [11]:
# 3. Create discount_amount.
# Formula: gross_amount × discount_pct / 100
discount_df = gross_df.withColumn ('discount_amount', gross_df.gross_amount * gross_df.discount_pct / 100)
discount_df.show(10, False)
discount_df.printSchema


+---------+----------+-----------+---------------+----------+------------------+-----------------+--------+-------------------------+-----------+-----+--------+----------+------------+------------+------------------+
|order_id |order_date|customer_id|customer_name  |product_id|product_name      |category         |store_id|store_name               |city       |state|quantity|unit_price|discount_pct|gross_amount|discount_amount   |
+---------+----------+-----------+---------------+----------+------------------+-----------------+--------+-------------------------+-----------+-----+--------+----------+------------+------------+------------------+
|ORD002629|NULL      |CUST00711  |Timothy Scott  |PROD00049 |Puzzle 1000pc     |Toys & Games     |STORE023|Las Vegas Mall Store     |Las Vegas  |NV   |9       |101.58    |0           |914.22      |0.0               |
|ORD004246|NULL      |CUST00138  |William Ramirez|PROD00105 |Board Game        |Toys & Games     |STORE013|Fort Worth Mall Store    

<bound method DataFrame.printSchema of DataFrame[order_id: string, order_date: double, customer_id: string, customer_name: string, product_id: string, product_name: string, category: string, store_id: string, store_name: string, city: string, state: string, quantity: int, unit_price: double, discount_pct: int, gross_amount: double, discount_amount: double]>

In [12]:
# 4. Create final_amount.
# Formula: gross_amount - discount_amount

final_df = discount_df.withColumn('final_amount', discount_df.gross_amount - discount_df.discount_amount)
final_df.show(10, False)


+---------+----------+-----------+---------------+----------+------------------+-----------------+--------+-------------------------+-----------+-----+--------+----------+------------+------------+------------------+------------+
|order_id |order_date|customer_id|customer_name  |product_id|product_name      |category         |store_id|store_name               |city       |state|quantity|unit_price|discount_pct|gross_amount|discount_amount   |final_amount|
+---------+----------+-----------+---------------+----------+------------------+-----------------+--------+-------------------------+-----------+-----+--------+----------+------------+------------+------------------+------------+
|ORD002629|NULL      |CUST00711  |Timothy Scott  |PROD00049 |Puzzle 1000pc     |Toys & Games     |STORE023|Las Vegas Mall Store     |Las Vegas  |NV   |9       |101.58    |0           |914.22      |0.0               |914.22      |
|ORD004246|NULL      |CUST00138  |William Ramirez|PROD00105 |Board Game        |

In [13]:
# 5. Rename columns.

# customer_id     → customer_key
# product_id      → product_key
# store_id        → store_key
# unit_price      → selling_price
# discount_pct    → discount_percentage


# withColumnRenamed --> changes the single column name at a time(we need to rewrite the same code multple times)
renamed_df = final_df.withColumnRenamed('customer_id', 'customer_key')\
                     .withColumnRenamed('product_id', 'product_key')\
                     .withColumnRenamed('store_id', 'store_key')\
                     .withColumnRenamed('unit_price', 'selling_price')\
                     .withColumnRenamed('discount_pct', 'discount_percentage')
                                                                                                                     # we Can use either one of these two functions

# withColumnsRenamed --> changes the smultiple column names at a time(no need to rewrite the same code multple times)
# pass the Dictionary(Key-value pairs)
renamed_df = final_df.withColumnsRenamed({'customer_id': 'customer_key',
                                          'product_id': 'product_key',
                                          'store_id': 'store_key',
                                          'unit_price': 'selling_price',
                                          'discount_pct': 'discount_percentage'})

renamed_df.show(10, False)


+---------+----------+------------+---------------+-----------+------------------+-----------------+---------+-------------------------+-----------+-----+--------+-------------+-------------------+------------+------------------+------------+
|order_id |order_date|customer_key|customer_name  |product_key|product_name      |category         |store_key|store_name               |city       |state|quantity|selling_price|discount_percentage|gross_amount|discount_amount   |final_amount|
+---------+----------+------------+---------------+-----------+------------------+-----------------+---------+-------------------------+-----------+-----+--------+-------------+-------------------+------------+------------------+------------+
|ORD002629|NULL      |CUST00711   |Timothy Scott  |PROD00049  |Puzzle 1000pc     |Toys & Games     |STORE023 |Las Vegas Mall Store     |Las Vegas  |NV   |9       |101.58       |0                  |914.22      |0.0               |914.22      |
|ORD004246|NULL      |CUST00

In [14]:
# 6. Create store-level sales summary.

# For every store calculate:

# total_orders
# total_quantity
# total_gross_sales
# total_discount
# total_sales
# average_sales
# minimum_sales
# maximum_sales


store_level_summary_df = renamed_df.groupBy('store_key').agg(
    count('order_id').alias('total_orders'),
    sum('quantity').alias('total_quantity'),
    sum('gross_amount').alias('total_gross_sales'),
    sum('discount_amount').alias('total_discount'),
    sum('final_amount').alias('total_sales'),
    avg('final_amount').alias('average_sales'),
    min('final_amount').alias('minimum_sales'),
    max('final_amount').alias('maximum_sales')
)
store_level_summary_df.show(10, False)


+---------+------------+--------------+------------------+------------------+------------------+-----------------+------------------+------------------+
|store_key|total_orders|total_quantity|total_gross_sales |total_discount    |total_sales       |average_sales    |minimum_sales     |maximum_sales     |
+---------+------------+--------------+------------------+------------------+------------------+-----------------+------------------+------------------+
|STORE004 |216         |1173          |164830.79         |18956.932000000004|145873.85799999992|675.3419351851849|6.734             |2372.9100000000003|
|STORE020 |177         |1031          |148836.61000000002|15598.648999999994|133237.961        |752.7568418079096|21.645            |2924.9            |
|STORE003 |220         |1201          |157978.22999999992|18359.291         |139618.93899999995|634.6315409090907|11.1225           |2909.9            |
|STORE012 |176         |981           |131515.15         |13577.542499999998|11793

In [15]:
# 7. Create category-level summary.

# For every category calculate:

# total_quantity
# total_sales
# average_sales

category_level_summary_df = renamed_df.groupBy('category').agg(
    sum('quantity').alias('total_quantity'),
    sum('final_amount').alias('total_sales'),
    avg('final_amount').alias('average_sales')
    )
category_level_summary_df.show(10, False)


+----------------------+--------------+------------------+-----------------+
|category              |total_quantity|total_sales       |average_sales    |
+----------------------+--------------+------------------+-----------------+
|Home & Kitchen        |3756          |480691.19150000036|696.6539007246382|
|Beauty & Personal Care|2085          |317425.97050000035|881.7388069444454|
|Toys & Games          |4535          |569535.4969999999 |666.1233883040934|
|Groceries             |3607          |402543.35550000024|628.9739929687504|
|Sports & Outdoors     |3399          |399179.7655000002 |646.9688257698544|
|Apparel               |4536          |489812.1649999999 |600.2600061274509|
|Books & Stationery    |3112          |402543.36949999986|729.245234601449 |
|Electronics           |2618          |307321.17900000024|653.8748489361707|
+----------------------+--------------+------------------+-----------------+



In [16]:
# 8. Create product-level summary.

# For every product calculate:

# total_quantity
# total_sales

# Sort products by highest sales.

product_level_summary_df = renamed_df.groupBy('product_name').agg(
    sum('quantity').alias('total_quantity'),
    sum('final_amount').alias('total_sales')
)
product_level_summary_df.show()


processed_sales_df =product_level_summary_df.sort(desc('total_sales'))
processed_sales_df.show(10, False)


+--------------------+--------------+------------------+
|        product_name|total_quantity|       total_sales|
+--------------------+--------------+------------------+
|         Ankle Socks|           513|56855.390999999996|
|          Power Bank|           191| 45520.76899999999|
|      Wireless Mouse|           501|        60303.4785|
|     Smartphone Case|           171|23386.096499999992|
|        Dumbbell Set|           161|31457.709499999997|
|        Mixing Bowls|           536| 70845.06999999999|
|    Wireless Earbuds|           132| 7162.724999999999|
|             Planner|           203|26948.194000000003|
|       Almond Butter|           404|         48461.491|
|  Whiteboard Markers|           194| 4526.246000000001|
|           Trail Mix|           395|27175.298000000006|
|           Honey Jar|           206|22342.461000000007|
|       Puzzle 1000pc|           914|119159.67800000001|
|     Sparkling Water|           935|        134184.659|
|         Rain Jacket|         

In [17]:
# 9. Write the processed transaction data.
# Output:
# processed_sales/
# Format: CSV
# 10. Write the store summary.
# Output:
# store_sales_summary/


# processed_sales_df.write.csv('processed_sales', header = True)

# Default Mode(error) --> if the folder does not exists, it will create it,
# otherwise it will throw an error as (AnalysisException: [PATH_ALREADY_EXISTS] Path file:/content/processed_sales already exists. Set mode as "overwrite" to overwrite the existing path. SQLSTATE: 42K04)
processed_sales_df.write.mode('overwrite').csv('processed_sales', header = True)


In [18]:

# overwrite mode:- --> if the folder does not exists, it will create it,
#otherwise it will throw an error as already exists, otherwise spark will overwrites the existing data in the path( old data gets deleted)

processed_sales_df.write.mode('overwrite').csv('processed_sales', header = True)

In [19]:
# Append Mode --> if the folder does not exist, it will create it; otherwise spark will append/adds new data to the existing path#
processed_sales_df.write.mode('append').csv('processed_sales', header = True)

In [20]:
# ignore Mode --> if the folder does not exist, it will create it; otherwise, Spark will ignore the current write
processed_sales_df.write.mode('append').csv('processed_sales', header = True)

In [21]:
processed_sales_df = product_level_summary_df.withColumn('total_sales', format_number(col('total_sales'), 2))
processed_sales_df = processed_sales_df.orderBy(desc('total_sales'))
processed_sales_df.show(10, False)

+-------------------+--------------+-----------+
|product_name       |total_quantity|total_sales|
+-------------------+--------------+-----------+
|Mechanical Keyboard|725           |95,675.72  |
|Yoga Pants         |591           |95,235.43  |
|Card Game          |550           |87,022.99  |
|Lip Balm           |563           |83,261.18  |
|Board Game         |756           |82,622.77  |
|Paper Clips        |788           |81,091.24  |
|Coffee Maker       |101           |8,705.46   |
|Non-Stick Pan      |189           |8,321.65   |
|Face Wash          |554           |78,911.53  |
|Hair Dryer         |459           |75,488.37  |
+-------------------+--------------+-----------+
only showing top 10 rows


***Spark write Modes:-***
1. Default Mode(error) --> if the folder does not exists, it will create it, otherwise it will throw an error as (AnalysisException: [PATH_ALREADY_EXISTS] Path file:/content/processed_sales already exists. Set mode as "overwrite" to overwrite the existing path. SQLSTATE: 42K04)

2. overwrite mode: --> if the folder does not exists, it will create it, otherwise it will throw an error as already exists, otherwise spark will overwrites the existing data in the path( old data gets deleted)

3. Append mode --> if the folder does not exists, it will create it, otherwise spark will append/adds new data to the existing path

4. ignore mode --> if the folder does not exists, it will create it, otherwise spark will ignore the current write

5. withColumn() --> if the column is not present in DF, It will create it, otherwisein the existing column itself the data will be overwritten

***difference btween file formats csv, parquet, ORC, avro***

-->
**CSV (Comma-Separated Values)**
Structure: Row-based, plain text, human-readable — each line is a record with comma-separated fields.
Schema: No built-in schema or data types; everything is treated as text/string.
Compression: Minimal to none natively; larger file sizes compared to binary formats.
Performance: Slow to read/write for large datasets — no indexing, no columnar pruning, must parse entire file.
Best use case: Small datasets, data exchange between systems, quick manual inspection, or when human readability matters.

**Parquet**

Structure: Columnar storage — data stored by column rather than row, ideal for analytics.
Schema: Strongly typed with embedded schema (supports nested/complex structures like structs, arrays, maps).
Compression: Excellent compression (Snappy, Gzip, Zstd) since similar column values compress well together.
Performance: Very fast for analytical queries (column pruning, predicate pushdown) but slower for row-by-row writes/updates.
Best use case: Big data analytics, data warehousing, Spark/Hive/Presto pipelines — the de facto standard for OLAP workloads.

**ORC (Optimized Row Columnar)**

Structure: Also columnar, but organizes data into "stripes" containing row groups, with rich metadata per stripe.
Schema: Strong schema support with type information, similar to Parquet, originally built for Hive.
Compression: Very high compression ratios (often better than Parquet for certain workloads), supports Zlib, Snappy.
Performance: Excellent for read-heavy Hive/Hadoop workloads; includes built-in indexes (min/max, bloom filters) for fast filtering.
Best use case: Hadoop/Hive-centric ecosystems, especially when ACID transactions or heavy predicate filtering are needed.

**Avro**

Structure: Row-based binary format — stores full records together, similar in spirit to CSV but binary and structured.
Schema: Schema is defined in JSON and stored with the data (or in a schema registry), enabling strong schema evolution support.
Compression: Good compression, though less efficient for analytical scans since it's row-oriented, not columnar.
Performance: Fast for write-heavy and streaming workloads; not ideal for scanning select columns across huge datasets.
Best use case: Data serialization for streaming/messaging systems (e.g., Kafka), and situations needing frequent schema changes over time.
